# Video Understanding: From Spatial-Temporal Attention to Dynamic-FPS Video-Language Models

**Papers**: Qwen2.5-VL (Dynamic FPS), Qwen3-VL (Text-Timestamp Alignment), ViViT (Arnab et al., 2021)

**Core challenge — token explosion in video:**

| Input | Computation | Tokens | Feasibility |
|---|---|---|---|
| 1 image | 256 patches | 256 | ✅ Manageable |
| 10s video @ 30fps | 300 frames × 256 patches | 76,800 | ❌ Exceeds context |
| 10s video @ dynamic FPS | 10-40 frames × 256 patches | 2,560-10,240 | ✅ Tractable |

**What we build in this notebook (all from scratch):**
1. Video loading with uniform, dynamic-FPS, and keyframe sampling
2. Factorized spatial-temporal attention (ViViT) and joint space-time attention
3. 3D mRoPE — extending rotary positional encoding to `(time, height, width)`
4. Token compression — temporal pooling, frame merging, spatial token merging
5. Text-timestamp alignment (Qwen3-VL style temporal grounding)
6. A complete mini video-language model: frames → ViT encoder → projector → LLM decoder
7. Video QA demo with temporal grounding

**Prerequisites:** Transformers, self-attention, ViT, RoPE basics.

---
# 0) Environment Setup

In [ ]:
!pip install torch torchvision matplotlib numpy einops pillow opencv-python-headless -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
from einops import rearrange, repeat
from typing import Optional, Tuple, List, Dict
from dataclasses import dataclass

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

---
# 1) Video Loading and Sampling Strategies

## 1.1 The Sampling Problem

Before any model can process video, we must decide **which frames** to feed it. This seemingly
simple choice has enormous impact on both compute cost and information retention.

**Three main strategies:**

| Strategy | Idea | When to use |
|---|---|---|
| **Uniform** | Sample every N-th frame | Baseline; simple but wasteful |
| **Dynamic FPS** | Adapt sampling rate to motion | Qwen2.5-VL; best compute/info tradeoff |
| **Keyframe** | Detect scene changes, sample boundaries | Long videos, surveillance |

**Sample input → output (Dynamic FPS):**
```
Input:  30fps raw video, 10 seconds = 300 frames
        frames 0-90:   lecture slide (low motion)
        frames 91-210: speaker gesturing (medium motion)
        frames 211-300: quick demo (high motion)

Output: [frame_0, frame_30, frame_60,    ← 1 FPS for low-motion
         frame_100, frame_115, frame_130, frame_145, frame_160, frame_175, frame_190, frame_205, ← ~7 FPS for medium
         frame_212, frame_218, frame_224, frame_230, frame_236, frame_242, frame_248,
         frame_254, frame_260, frame_266, frame_272, frame_278, frame_284, frame_290, frame_296] ← ~15 FPS for high
Total:  28 frames instead of 300 → 10.7× reduction
```

## 1.2 Synthetic Video Generation

We generate synthetic videos with controllable motion for reproducible experiments.
Each video contains segments of varying dynamics — slow panning, static, fast motion —
so we can verify that dynamic FPS correctly adapts its sampling rate.

In [ ]:
def generate_synthetic_video(
    num_frames: int = 120,
    height: int = 64,
    width: int = 64,
    channels: int = 3,
) -> torch.Tensor:
    """Generate a synthetic video with three motion regimes.

    Returns:
        # (num_frames, channels, height, width) — pixel values in [0, 1]

    The video has three segments:
      - Segment 1 (0 to num_frames//3):     Slow — a circle drifts slowly
      - Segment 2 (num_frames//3 to 2*num_frames//3): Static — frozen frame
      - Segment 3 (2*num_frames//3 to end):  Fast — rapid color and shape changes
    """
    video = torch.zeros(num_frames, channels, height, width)

    seg1_end = num_frames // 3
    seg2_end = 2 * num_frames // 3

    yy, xx = torch.meshgrid(
        torch.arange(height, dtype=torch.float32),
        torch.arange(width, dtype=torch.float32),
        indexing="ij",
    )

    # Segment 1: slow-moving circle
    for t in range(seg1_end):
        cx = width / 2 + 5 * math.sin(2 * math.pi * t / seg1_end)
        cy = height / 2 + 5 * math.cos(2 * math.pi * t / seg1_end)
        dist = torch.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        mask = (dist < 10).float()
        video[t, 0] = mask * 0.9
        video[t, 1] = mask * 0.3
        video[t, 2] = 0.1

    # Segment 2: static frame (copy the last frame of segment 1)
    for t in range(seg1_end, seg2_end):
        video[t] = video[seg1_end - 1]

    # Segment 3: fast random motion
    for t in range(seg2_end, num_frames):
        speed = (t - seg2_end) / (num_frames - seg2_end)
        cx = width / 2 + 20 * math.sin(8 * math.pi * speed)
        cy = height / 2 + 20 * math.cos(12 * math.pi * speed)
        dist = torch.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        radius = 5 + 10 * abs(math.sin(6 * math.pi * speed))
        mask = (dist < radius).float()
        video[t, 0] = mask * abs(math.sin(4 * math.pi * speed))
        video[t, 1] = mask * abs(math.cos(3 * math.pi * speed))
        video[t, 2] = (1 - mask) * 0.15 + mask * abs(math.sin(7 * math.pi * speed))

    return video


# Generate a test video
synth_video = generate_synthetic_video(num_frames=120, height=64, width=64)
print(f"Synthetic video shape: {synth_video.shape}")
print(f"  → (num_frames={synth_video.shape[0]}, C={synth_video.shape[1]}, "
      f"H={synth_video.shape[2]}, W={synth_video.shape[3]})")

In [ ]:
# Visualize selected frames from each motion segment
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
sample_indices = [0, 20, 45, 65, 90, 115]
for ax, idx in zip(axes, sample_indices):
    frame = synth_video[idx].permute(1, 2, 0).numpy()
    ax.imshow(frame.clip(0, 1))
    segment = "slow" if idx < 40 else ("static" if idx < 80 else "fast")
    ax.set_title(f"Frame {idx}\n({segment})", fontsize=9)
    ax.axis("off")
plt.suptitle("Synthetic Video — Three Motion Regimes", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 1.3 Uniform Sampling

The simplest strategy: pick every N-th frame. Fast, deterministic, but oblivious to content.
A 30fps lecture and a 30fps sports clip get the same treatment.

In [ ]:
def uniform_sample(
    video: torch.Tensor, num_samples: int
) -> Tuple[torch.Tensor, List[int]]:
    """Uniformly sample frames from a video.

    Args:
        video: (num_frames, C, H, W)
        num_samples: how many frames to select

    Returns:
        sampled_frames: (num_samples, C, H, W)
        indices: list of selected frame indices
    """
    num_frames = video.shape[0]
    indices = np.linspace(0, num_frames - 1, num_samples, dtype=int).tolist()
    return video[indices], indices


uniform_frames, uniform_idx = uniform_sample(synth_video, num_samples=8)
print(f"Uniform sampling: selected frames {uniform_idx}")
print(f"Sampled shape: {uniform_frames.shape}")

## 1.4 Dynamic FPS Sampling (Qwen2.5-VL)

**Key insight from Qwen2.5-VL:** Not all temporal segments carry equal information.
A static lecture slide needs 1 FPS; a fast sports replay needs 8+ FPS.

**Algorithm:**
1. Compute inter-frame difference (pixel-level L1 or optical flow magnitude)
2. Smooth the motion signal to avoid noise-driven spikes
3. Map motion magnitude → local FPS via a configurable curve
4. Accumulate time until the next sample is due, then grab that frame

This gives us a **variable-length** frame sequence that concentrates tokens on dynamic segments.

```
Motion signal:     ▁▁▁▁▁▂▂▂▃▃▅▅▇▇█████▇▅▃▂▁▁▁
Sampled density:   ·  ·  ·  · ·· ···········  ·  ·
                   ←—low fps—→←——high fps——→←—low—→
```

In [ ]:
def compute_motion_signal(video: torch.Tensor) -> torch.Tensor:
    """Compute per-frame motion magnitude via inter-frame L1 difference.

    Args:
        video: (num_frames, C, H, W) — pixel values

    Returns:
        motion: (num_frames,) — motion score per frame (first frame = 0)
    """
    num_frames = video.shape[0]
    motion = torch.zeros(num_frames)

    # L1 difference between consecutive frames, averaged over all pixels
    # (num_frames-1, C, H, W) → (num_frames-1,) after mean
    diffs = (video[1:] - video[:-1]).abs().mean(dim=(1, 2, 3))
    motion[1:] = diffs

    return motion


def smooth_signal(signal: torch.Tensor, kernel_size: int = 5) -> torch.Tensor:
    """Apply 1D moving-average smoothing to reduce noise.

    Args:
        signal: (num_frames,)

    Returns:
        smoothed: (num_frames,) — same length, edges handled by padding
    """
    kernel = torch.ones(kernel_size) / kernel_size
    padded = F.pad(signal.unsqueeze(0).unsqueeze(0), (kernel_size // 2, kernel_size // 2), mode="reflect")
    # (1, 1, num_frames + pad) → (1, 1, num_frames)
    smoothed = F.conv1d(padded, kernel.unsqueeze(0).unsqueeze(0))
    return smoothed.squeeze()


motion = compute_motion_signal(synth_video)
motion_smooth = smooth_signal(motion, kernel_size=7)

plt.figure(figsize=(12, 3))
plt.plot(motion.numpy(), alpha=0.4, label="Raw motion")
plt.plot(motion_smooth.numpy(), linewidth=2, label="Smoothed motion")
plt.axvline(x=40, color="gray", linestyle="--", alpha=0.5, label="Segment boundaries")
plt.axvline(x=80, color="gray", linestyle="--", alpha=0.5)
plt.xlabel("Frame index")
plt.ylabel("Motion magnitude")
plt.title("Inter-frame Motion Signal")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def dynamic_fps_sample(
    video: torch.Tensor,
    min_fps: float = 0.5,
    max_fps: float = 8.0,
    original_fps: float = 30.0,
    smooth_kernel: int = 7,
) -> Tuple[torch.Tensor, List[int], torch.Tensor]:
    """Sample frames using content-adaptive dynamic FPS (Qwen2.5-VL style).

    The local FPS is linearly interpolated between min_fps and max_fps based on
    the normalized motion magnitude at each frame.

    Args:
        video: (num_frames, C, H, W)
        min_fps: FPS for zero-motion segments
        max_fps: FPS for maximum-motion segments
        original_fps: the native FPS of the input video
        smooth_kernel: kernel size for motion smoothing

    Returns:
        sampled_frames: (num_sampled, C, H, W)
        indices: list of selected frame indices
        local_fps: (num_frames,) — the computed FPS at each frame
    """
    num_frames = video.shape[0]

    motion = compute_motion_signal(video)
    motion = smooth_signal(motion, kernel_size=smooth_kernel)

    # Normalize motion to [0, 1]
    m_min, m_max = motion.min(), motion.max()
    if m_max - m_min > 1e-8:
        motion_norm = (motion - m_min) / (m_max - m_min)
    else:
        motion_norm = torch.zeros_like(motion)

    # Map normalized motion → local FPS
    local_fps = min_fps + (max_fps - min_fps) * motion_norm

    # Walk through frames, accumulating time until next sample is due
    indices = [0]
    accumulated_time = 0.0
    for i in range(1, num_frames):
        # Time elapsed between frame i-1 and frame i at the original FPS
        dt = 1.0 / original_fps
        accumulated_time += dt
        # Time between samples at the local FPS
        sample_interval = 1.0 / local_fps[i].item()
        if accumulated_time >= sample_interval:
            indices.append(i)
            accumulated_time = 0.0

    sampled_frames = video[indices]
    return sampled_frames, indices, local_fps


dyn_frames, dyn_idx, local_fps = dynamic_fps_sample(
    synth_video, min_fps=0.5, max_fps=8.0, original_fps=30.0
)

print(f"Dynamic FPS selected {len(dyn_idx)} frames out of {synth_video.shape[0]}")
print(f"Compression ratio: {synth_video.shape[0] / len(dyn_idx):.1f}×")

# Analyze per-segment density
seg1 = [i for i in dyn_idx if i < 40]
seg2 = [i for i in dyn_idx if 40 <= i < 80]
seg3 = [i for i in dyn_idx if i >= 80]
print(f"Segment densities — slow: {len(seg1)} frames, static: {len(seg2)} frames, fast: {len(seg3)} frames")

In [ ]:
# Visualize the dynamic FPS sampling pattern
fig, axes = plt.subplots(2, 1, figsize=(14, 5), gridspec_kw={"height_ratios": [1, 2]})

axes[0].plot(local_fps.numpy(), color="tab:blue", linewidth=1.5)
axes[0].set_ylabel("Local FPS")
axes[0].set_title("Dynamic FPS — Sampling Density Adapts to Motion")
axes[0].axvline(x=40, color="gray", linestyle="--", alpha=0.5)
axes[0].axvline(x=80, color="gray", linestyle="--", alpha=0.5)

# Show selected frame positions as a rug plot
for idx in dyn_idx:
    axes[1].axvline(x=idx, color="tab:red", alpha=0.6, linewidth=0.8)
axes[1].set_xlabel("Frame index")
axes[1].set_ylabel("Selected")
axes[1].set_title(f"Selected Frames ({len(dyn_idx)} total)")
axes[1].set_yticks([])

plt.tight_layout()
plt.show()

## 1.5 Keyframe / Scene-Change Sampling

For very long videos (hours of surveillance, full movies), we detect **scene boundaries**
where the visual content changes abruptly. Only boundary frames plus periodic "heartbeat"
frames are retained. This is complementary to dynamic FPS — useful when most of the video
is truly redundant.

In [ ]:
def keyframe_sample(
    video: torch.Tensor,
    threshold: float = 0.05,
    min_interval: int = 5,
    max_interval: int = 30,
) -> Tuple[torch.Tensor, List[int]]:
    """Sample keyframes at scene changes plus periodic heartbeats.

    Args:
        video: (num_frames, C, H, W)
        threshold: motion threshold to trigger a keyframe
        min_interval: minimum frames between keyframes (suppress rapid fire)
        max_interval: maximum gap — forces a heartbeat frame

    Returns:
        sampled_frames: (num_sampled, C, H, W)
        indices: list of keyframe indices
    """
    motion = compute_motion_signal(video)
    motion = smooth_signal(motion, kernel_size=5)

    indices = [0]
    for i in range(1, len(video)):
        gap = i - indices[-1]
        is_scene_change = motion[i].item() > threshold and gap >= min_interval
        is_heartbeat = gap >= max_interval
        if is_scene_change or is_heartbeat:
            indices.append(i)

    return video[indices], indices


kf_frames, kf_idx = keyframe_sample(synth_video, threshold=0.02)
print(f"Keyframe sampling: {len(kf_idx)} frames selected")
print(f"Indices: {kf_idx}")

---
# 2) Extending ViT to Video: Spatial-Temporal Attention

## 2.1 From Image Patches to Video Tubes

A Vision Transformer (ViT) converts an image into a sequence of patch embeddings:
```
Image (C, H, W) → patches (num_patches, patch_dim) → linear projection → (num_patches, model_dim)
```

For video, we have an additional **temporal** axis. Two main approaches from ViViT (Arnab et al., 2021):

| Approach | Attention pattern | Complexity | Quality |
|---|---|---|---|
| **Factorized** (Model 2/3) | Spatial-only then temporal-only | O(T·N² + N·T²) | Good, efficient |
| **Joint** (Model 1) | All patches attend to all patches across all frames | O((T·N)²) | Best, expensive |

Where T = number of frames, N = patches per frame.

**Factorized attention visual:**
```
Frame 1:  [p₁₁ p₁₂ ... p₁ₙ] ──► Spatial Self-Attention (within frame)
Frame 2:  [p₂₁ p₂₂ ... p₂ₙ] ──► Spatial Self-Attention (within frame)
  ...
Frame T:  [pₜ₁ pₜ₂ ... pₜₙ] ──► Spatial Self-Attention (within frame)

Then for each spatial position j:
  [p₁ⱼ, p₂ⱼ, ..., pₜⱼ] ──► Temporal Self-Attention (across frames)
```

**Why factorize?** Joint attention over T×N tokens has complexity O((TN)²).
Factorized attention is O(T·N² + N·T²) — much cheaper when both T and N are large.

## 2.2 Video Patch Embedding

First, we need to convert raw video frames into patch token sequences. Each frame is split
into a grid of non-overlapping patches, then linearly projected to `model_dim`.

In [ ]:
class VideoPatchEmbedding(nn.Module):
    """Convert video frames into a sequence of patch embeddings.

    Input:  (batch_num, num_frames, C, H, W)
    Output: (batch_num, num_frames, num_patches_per_frame, model_dim)

    Each frame is independently split into non-overlapping patches and projected.
    This is equivalent to a 2D convolution with kernel_size=stride=patch_size.
    """

    def __init__(self, img_size: int = 64, patch_size: int = 8, in_channels: int = 3, model_dim: int = 256):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches_h = img_size // patch_size
        self.num_patches_w = img_size // patch_size
        self.num_patches = self.num_patches_h * self.num_patches_w
        patch_dim = in_channels * patch_size * patch_size

        # Linear projection from flattened patch pixels to model_dim
        # (patch_dim,) → (model_dim,)
        self.proj = nn.Linear(patch_dim, model_dim)
        self.norm = nn.LayerNorm(model_dim)

    def forward(self, video: torch.Tensor) -> torch.Tensor:
        """
        Args:
            video: (batch_num, num_frames, C, H, W)
        Returns:
            patch_embeddings: (batch_num, num_frames, num_patches, model_dim)
        """
        B, T, C, H, W = video.shape
        ps = self.patch_size

        # Reshape into patches: (B, T, C, H, W) → (B, T, nH, ps, nW, ps, C) → (B, T, nH*nW, ps*ps*C)
        x = video.reshape(B, T, C, self.num_patches_h, ps, self.num_patches_w, ps)
        x = x.permute(0, 1, 3, 5, 2, 4, 6)
        # (batch_num, num_frames, nH, nW, C, ps, ps)
        x = x.reshape(B, T, self.num_patches, -1)
        # (batch_num, num_frames, num_patches, patch_dim) → (batch_num, num_frames, num_patches, model_dim)
        x = self.norm(self.proj(x))
        return x


# Test
patch_embed = VideoPatchEmbedding(img_size=64, patch_size=8, in_channels=3, model_dim=256)
test_video = synth_video[:16].unsqueeze(0)  # (1, 16, 3, 64, 64)
patches = patch_embed(test_video)
print(f"Input video:     {test_video.shape}  → (batch_num, num_frames, C, H, W)")
print(f"Patch embeddings: {patches.shape}  → (batch_num, num_frames, num_patches, model_dim)")
print(f"Patches per frame: {patch_embed.num_patches} ({patch_embed.num_patches_h}×{patch_embed.num_patches_w})")

## 2.3 Multi-Head Self-Attention (Building Block)

Both factorized and joint approaches use the same MHSA core. We implement it once
and reuse it with different reshaping strategies.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Standard multi-head self-attention.

    Input:  (batch_num, seq_len, model_dim)
    Output: (batch_num, seq_len, model_dim)
    """

    def __init__(self, model_dim: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert model_dim % num_heads == 0, "model_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Combined QKV projection for efficiency
        # (model_dim,) → (3 * model_dim,)
        self.qkv = nn.Linear(model_dim, 3 * model_dim)
        self.out_proj = nn.Linear(model_dim, model_dim)
        self.attn_drop = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x: (batch_num, seq_len, model_dim)
            attn_mask: optional (batch_num, num_heads, seq_len, seq_len)

        Returns:
            output: (batch_num, seq_len, model_dim)
            attn_weights: (batch_num, num_heads, seq_len, seq_len)
        """
        B, N, D = x.shape

        # Project to Q, K, V
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, 3 * model_dim)
        qkv = self.qkv(x)
        # (batch_num, seq_len, 3, num_heads, head_dim)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim)
        # (3, batch_num, num_heads, seq_len, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        # Scaled dot-product attention
        # (batch_num, num_heads, seq_len, head_dim) @ (batch_num, num_heads, head_dim, seq_len)
        # → (batch_num, num_heads, seq_len, seq_len)
        attn = (q @ k.transpose(-2, -1)) * self.scale

        if attn_mask is not None:
            attn = attn + attn_mask

        attn_weights = F.softmax(attn, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        # (batch_num, num_heads, seq_len, seq_len) @ (batch_num, num_heads, seq_len, head_dim)
        # → (batch_num, num_heads, seq_len, head_dim)
        out = attn_weights @ v
        # (batch_num, seq_len, model_dim)
        out = out.transpose(1, 2).reshape(B, N, D)
        out = self.out_proj(out)

        return out, attn_weights

## 2.4 Factorized Spatial-Temporal Attention (ViViT Model 2)

The key idea: **decompose** the expensive joint space-time attention into two cheaper passes.

**Pass 1 — Spatial:** Each frame's patches attend only to other patches in the **same frame**.
This captures spatial relationships (object shapes, textures) independently per frame.

**Pass 2 — Temporal:** For each spatial position, tokens at that position across **all frames**
attend to each other. This captures how each spatial location evolves over time.

```
Input:  (B, T, N, D)     ← T frames, N patches each

Spatial pass:
  reshape → (B*T, N, D)  ← treat each frame as a separate sequence
  MHSA    → (B*T, N, D)
  reshape → (B, T, N, D)

Temporal pass:
  transpose → (B*N, T, D) ← treat each spatial position as a separate sequence
  MHSA      → (B*N, T, D)
  transpose → (B, T, N, D)
```

In [ ]:
class FactorizedTransformerBlock(nn.Module):
    """One block of factorized spatial-temporal attention (ViViT Model 2).

    Input:  (batch_num, num_frames, num_patches, model_dim)
    Output: (batch_num, num_frames, num_patches, model_dim)

    Design choice: spatial-first then temporal. Arnab et al. found this ordering
    slightly better than temporal-first, likely because spatial coherence within a
    frame is a stronger prior than temporal coherence across frames.
    """

    def __init__(self, model_dim: int, num_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        # Spatial attention (within each frame)
        self.norm_s1 = nn.LayerNorm(model_dim)
        self.spatial_attn = MultiHeadSelfAttention(model_dim, num_heads, dropout)

        # Temporal attention (across frames, per spatial position)
        self.norm_t1 = nn.LayerNorm(model_dim)
        self.temporal_attn = MultiHeadSelfAttention(model_dim, num_heads, dropout)

        # Shared feed-forward network
        mlp_dim = int(model_dim * mlp_ratio)
        self.norm_ff = nn.LayerNorm(model_dim)
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, model_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Args:
            x: (batch_num, num_frames, num_patches, model_dim)
        Returns:
            x: (batch_num, num_frames, num_patches, model_dim)
            attn_info: dict with spatial and temporal attention weights
        """
        B, T, N, D = x.shape

        # Spatial self-attention: each frame independently
        # (batch_num, num_frames, num_patches, model_dim) → (batch_num * num_frames, num_patches, model_dim)
        x_s = self.norm_s1(x)
        x_s = x_s.reshape(B * T, N, D)
        spatial_out, spatial_attn = self.spatial_attn(x_s)
        spatial_out = spatial_out.reshape(B, T, N, D)
        x = x + spatial_out

        # Temporal self-attention: each spatial position independently
        # (batch_num, num_frames, num_patches, model_dim) → (batch_num * num_patches, num_frames, model_dim)
        x_t = self.norm_t1(x)
        x_t = x_t.permute(0, 2, 1, 3).reshape(B * N, T, D)
        temporal_out, temporal_attn = self.temporal_attn(x_t)
        temporal_out = temporal_out.reshape(B, N, T, D).permute(0, 2, 1, 3)
        x = x + temporal_out

        # Feed-forward network
        x = x + self.ffn(self.norm_ff(x))

        attn_info = {
            "spatial": spatial_attn.reshape(B, T, -1, N, N),
            "temporal": temporal_attn.reshape(B, N, -1, T, T),
        }
        return x, attn_info


# Test factorized block
block = FactorizedTransformerBlock(model_dim=256, num_heads=8)
test_input = torch.randn(2, 8, 64, 256)  # (batch_num=2, num_frames=8, num_patches=64, model_dim=256)
out, attn = block(test_input)
print(f"Factorized block:")
print(f"  Input:  {test_input.shape} → (batch_num, num_frames, num_patches, model_dim)")
print(f"  Output: {out.shape}")
print(f"  Spatial attn:  {attn['spatial'].shape} → (B, T, heads, N, N)")
print(f"  Temporal attn: {attn['temporal'].shape} → (B, N, heads, T, T)")

## 2.5 Joint Space-Time Attention (ViViT Model 1)

The alternative: **flatten all patches from all frames** into one long sequence and let
full self-attention discover both spatial and temporal relationships.

```
Input:  (B, T, N, D)
Flatten: (B, T*N, D)    ← every patch can attend to every other patch across all frames
MHSA:   (B, T*N, D)
Reshape: (B, T, N, D)
```

**Pros:** Maximum expressivity — the model can learn arbitrary space-time interactions.
**Cons:** Quadratic in T×N, which is why Qwen2.5-VL uses dynamic FPS to keep T×N manageable.

In [ ]:
class JointSpaceTimeBlock(nn.Module):
    """Joint space-time attention block (ViViT Model 1).

    Input:  (batch_num, num_frames, num_patches, model_dim)
    Output: (batch_num, num_frames, num_patches, model_dim)

    All T×N tokens attend to each other in a single MHSA pass.
    Complexity: O((T*N)²) — use only when T*N is manageable.
    """

    def __init__(self, model_dim: int, num_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(model_dim)
        self.attn = MultiHeadSelfAttention(model_dim, num_heads, dropout)

        mlp_dim = int(model_dim * mlp_ratio)
        self.norm2 = nn.LayerNorm(model_dim)
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, model_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x: (batch_num, num_frames, num_patches, model_dim)
        Returns:
            x: (batch_num, num_frames, num_patches, model_dim)
            attn_weights: (batch_num, num_heads, T*N, T*N)
        """
        B, T, N, D = x.shape

        # Flatten space and time into one sequence
        # (batch_num, num_frames * num_patches, model_dim)
        x_flat = self.norm1(x).reshape(B, T * N, D)
        attn_out, attn_weights = self.attn(x_flat)
        x = x + attn_out.reshape(B, T, N, D)

        x = x + self.ffn(self.norm2(x))
        return x, attn_weights


# Test joint block
joint_block = JointSpaceTimeBlock(model_dim=256, num_heads=8)
test_input = torch.randn(2, 8, 64, 256)
out_j, attn_j = joint_block(test_input)
print(f"Joint space-time block:")
print(f"  Input:  {test_input.shape}")
print(f"  Output: {out_j.shape}")
print(f"  Attention: {attn_j.shape} → (B, heads, T*N, T*N)")
print(f"  Attention matrix size: {8*64}×{8*64} = {(8*64)**2:,} entries")

## 2.6 Empirical Complexity Comparison

In [ ]:
import time

def benchmark_attention(model, video_tokens, label, num_runs=10):
    """Measure forward pass time for an attention block."""
    # Warmup
    with torch.no_grad():
        for _ in range(3):
            model(video_tokens)

    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.time()
            model(video_tokens)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append(time.time() - start)
    avg = np.mean(times) * 1000
    std = np.std(times) * 1000
    print(f"  {label}: {avg:.1f} ± {std:.1f} ms")
    return avg

print("Benchmarking factorized vs joint attention:")
print("=" * 55)
configs = [(8, 64), (16, 64), (16, 128)]
for T, N in configs:
    print(f"\nT={T} frames, N={N} patches (sequence length = {T*N}):")
    tokens = torch.randn(1, T, N, 256)
    fact = FactorizedTransformerBlock(256, 8)
    joint = JointSpaceTimeBlock(256, 8)
    t_fact = benchmark_attention(fact, tokens, "Factorized")
    t_joint = benchmark_attention(joint, tokens, "Joint     ")
    print(f"  Speedup: {t_joint / t_fact:.2f}×")

---
# 3) Multi-dimensional Rotary Position Embedding (mRoPE) for Video

## 3.1 From 1D RoPE to 3D mRoPE

Standard RoPE encodes a **1D position** (token index in a sequence) as rotations applied to
query/key vectors. For video, each token has a **3D position**: `(time, height, width)`.

**Qwen2.5-VL's mRoPE** splits the embedding dimension into three equal parts and applies
independent rotary encodings to each:

```
model_dim = 256, split into 3 groups (round to nearest):
  dims [0:84]    ← encode temporal position (frame index)
  dims [84:170]  ← encode spatial height (patch row)
  dims [170:256] ← encode spatial width (patch column)

For an image (static): time=0 for all patches → temporal RoPE is identity
For video: time=frame_idx → encodes temporal order naturally
```

**Why this works:** Rotary embeddings are multiplicative, so the three orthogonal position
axes don't interfere with each other. The model can learn to route spatial vs temporal
queries to the appropriate dimension subspaces.

```
Standard RoPE (1D):
  q[2i], q[2i+1]  ← rotated by angle θᵢ * position

mRoPE (3D):
  q[0:84]    ← rotated by θᵢ * t    (time)
  q[84:170]  ← rotated by θᵢ * h    (height)
  q[170:256] ← rotated by θᵢ * w    (width)
```

In [ ]:
class VideoMRoPE(nn.Module):
    """Multi-dimensional Rotary Position Embedding for video (Qwen2.5-VL style).

    Splits the head dimension into three groups and applies 1D RoPE independently
    for time, height, and width coordinates.

    The rotation angles follow the standard RoPE formula:
        θᵢ = base^(-2i / d_group) where d_group is the dimension of each group.
    """

    def __init__(self, head_dim: int, base: float = 10000.0):
        super().__init__()
        self.head_dim = head_dim

        # Split head_dim into 3 roughly equal groups for (time, height, width)
        # Each group must have even dimension for the (sin, cos) rotation pairs
        group_size = head_dim // 3
        if group_size % 2 != 0:
            group_size -= 1
        self.d_time = group_size
        self.d_height = group_size
        self.d_width = head_dim - 2 * group_size
        if self.d_width % 2 != 0:
            self.d_width -= 1

        # Precompute inverse frequency for each group
        # θᵢ = base^(-2i / d) for i = 0, 1, ..., d/2 - 1
        self.register_buffer("inv_freq_t", self._make_inv_freq(self.d_time, base))
        self.register_buffer("inv_freq_h", self._make_inv_freq(self.d_height, base))
        self.register_buffer("inv_freq_w", self._make_inv_freq(self.d_width, base))

    @staticmethod
    def _make_inv_freq(dim: int, base: float) -> torch.Tensor:
        """Compute inverse frequencies: base^(-2i/dim) for i in [0, dim/2)."""
        return 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))

    def _apply_rotary(
        self, x: torch.Tensor, positions: torch.Tensor, inv_freq: torch.Tensor
    ) -> torch.Tensor:
        """Apply rotary embedding to a subset of dimensions.

        Args:
            x: (..., dim) — the slice of q or k to rotate
            positions: (...) — integer or float positions for each token
            inv_freq: (dim/2,) — the frequency basis

        Returns:
            rotated: (..., dim) — same shape as x, with rotary applied
        """
        # Compute angles: position * inverse_frequency
        # (..., 1) * (dim/2,) → (..., dim/2)
        angles = positions.unsqueeze(-1).float() * inv_freq.to(x.device)

        cos_a = angles.cos()
        sin_a = angles.sin()

        # Split x into pairs and apply rotation
        # (..., dim) → (..., dim/2, 2)
        x_pairs = x.reshape(*x.shape[:-1], -1, 2)
        x0 = x_pairs[..., 0]
        x1 = x_pairs[..., 1]

        # Standard rotary: [x0*cos - x1*sin, x0*sin + x1*cos]
        out0 = x0 * cos_a - x1 * sin_a
        out1 = x0 * sin_a + x1 * cos_a

        return torch.stack([out0, out1], dim=-1).flatten(-2)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        time_ids: torch.Tensor,
        height_ids: torch.Tensor,
        width_ids: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Apply 3D mRoPE to queries and keys.

        Args:
            q: (batch_num, num_heads, seq_len, head_dim)
            k: (batch_num, num_heads, seq_len, head_dim)
            time_ids:   (batch_num, seq_len) — frame index for each token
            height_ids: (batch_num, seq_len) — patch row for each token
            width_ids:  (batch_num, seq_len) — patch column for each token

        Returns:
            q_rot, k_rot: same shapes as input, with mRoPE applied
        """
        dt = self.d_time
        dh = self.d_height
        dw = self.d_width

        # Split into three groups along head_dim
        q_t, q_h, q_w, q_rest = q[..., :dt], q[..., dt:dt+dh], q[..., dt+dh:dt+dh+dw], q[..., dt+dh+dw:]
        k_t, k_h, k_w, k_rest = k[..., :dt], k[..., dt:dt+dh], k[..., dt+dh:dt+dh+dw], k[..., dt+dh+dw:]

        # Expand position IDs for broadcasting over num_heads
        # (batch_num, seq_len) → (batch_num, 1, seq_len) for broadcasting
        t = time_ids.unsqueeze(1).expand(-1, q.shape[1], -1)
        h = height_ids.unsqueeze(1).expand(-1, q.shape[1], -1)
        w = width_ids.unsqueeze(1).expand(-1, q.shape[1], -1)

        # Apply rotary to each group independently
        q_t = self._apply_rotary(q_t, t, self.inv_freq_t)
        q_h = self._apply_rotary(q_h, h, self.inv_freq_h)
        q_w = self._apply_rotary(q_w, w, self.inv_freq_w)

        k_t = self._apply_rotary(k_t, t, self.inv_freq_t)
        k_h = self._apply_rotary(k_h, h, self.inv_freq_h)
        k_w = self._apply_rotary(k_w, w, self.inv_freq_w)

        # Reassemble
        q_rot = torch.cat([q_t, q_h, q_w, q_rest], dim=-1)
        k_rot = torch.cat([k_t, k_h, k_w, k_rest], dim=-1)

        return q_rot, k_rot


# Test mRoPE
head_dim = 64
mrope = VideoMRoPE(head_dim=head_dim)
print(f"mRoPE dimension split: time={mrope.d_time}, height={mrope.d_height}, width={mrope.d_width}")
print(f"Total used: {mrope.d_time + mrope.d_height + mrope.d_width}/{head_dim}")

B, H, T, N_h, N_w = 1, 4, 8, 8, 8
N = N_h * N_w  # patches per frame
S = T * N      # total sequence length

q_test = torch.randn(B, H, S, head_dim)
k_test = torch.randn(B, H, S, head_dim)

# Build position IDs for each token: (time, height, width)
time_ids = torch.arange(T).repeat_interleave(N).unsqueeze(0)
height_ids = torch.arange(N_h).repeat(N_w).repeat(T).unsqueeze(0)
width_ids = torch.arange(N_w).repeat_interleave(1).repeat(N_h).repeat(T).unsqueeze(0)

print(f"\nSequence length: {S} (T={T} × N={N})")
print(f"time_ids sample:   {time_ids[0, :12].tolist()} ...")
print(f"height_ids sample: {height_ids[0, :12].tolist()} ...")

q_rot, k_rot = mrope(q_test, k_test, time_ids, height_ids, width_ids)
print(f"\nq before mRoPE norm: {q_test.norm(dim=-1).mean():.4f}")
print(f"q after  mRoPE norm: {q_rot.norm(dim=-1).mean():.4f}")
print(f"(Rotary preserves vector norm — this is a key property)")

## 3.2 Visualizing mRoPE Attention Patterns

mRoPE creates a **position-dependent bias** in attention scores. Tokens that are nearby
in (time, height, width) space naturally attend more to each other. Let's visualize this.

In [ ]:
def compute_mrope_bias(mrope_module, time_ids, height_ids, width_ids, head_dim, num_heads=1):
    """Compute the attention bias introduced by mRoPE for visualization.

    Uses identity vectors so the only contribution to attention scores
    comes from the positional rotations.
    """
    S = time_ids.shape[1]
    B = 1

    # Use unit vectors so attention = purely positional
    q = torch.zeros(B, num_heads, S, head_dim)
    k = torch.zeros(B, num_heads, S, head_dim)
    # Set all dims to 1/sqrt(head_dim) for normalized dot product
    q[:] = 1.0 / math.sqrt(head_dim)
    k[:] = 1.0 / math.sqrt(head_dim)

    q_rot, k_rot = mrope_module(q, k, time_ids, height_ids, width_ids)

    # Compute attention scores
    # (B, num_heads, S, S)
    scores = torch.matmul(q_rot, k_rot.transpose(-2, -1))
    return scores[0, 0]  # (S, S)


# Small example: 4 frames, 4×4 patches = 16 patches per frame, total 64 tokens
T_vis, Nh_vis, Nw_vis = 4, 4, 4
N_vis = Nh_vis * Nw_vis
S_vis = T_vis * N_vis

t_ids = torch.arange(T_vis).repeat_interleave(N_vis).unsqueeze(0)
h_ids = (torch.arange(N_vis) // Nw_vis).repeat(T_vis).unsqueeze(0)
w_ids = (torch.arange(N_vis) % Nw_vis).repeat(T_vis).unsqueeze(0)

mrope_vis = VideoMRoPE(head_dim=48)
bias = compute_mrope_bias(mrope_vis, t_ids, h_ids, w_ids, head_dim=48)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(bias.detach().numpy(), cmap="RdBu_r", aspect="auto")
axes[0].set_title(f"mRoPE Attention Bias ({T_vis} frames × {N_vis} patches)", fontsize=11)
axes[0].set_xlabel("Key position")
axes[0].set_ylabel("Query position")
for i in range(1, T_vis):
    axes[0].axhline(y=i * N_vis - 0.5, color="black", linewidth=0.5)
    axes[0].axvline(x=i * N_vis - 0.5, color="black", linewidth=0.5)
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Show bias for one query token (center of frame 0) vs all keys
query_idx = N_vis // 2 + Nw_vis // 2  # center patch of frame 0
bias_row = bias[query_idx].reshape(T_vis, Nh_vis, Nw_vis)
frames_to_show = min(T_vis, 4)
combined = torch.cat([bias_row[t] for t in range(frames_to_show)], dim=1)
im1 = axes[1].imshow(combined.detach().numpy(), cmap="RdBu_r", aspect="auto")
axes[1].set_title(f"Bias from query at frame 0 center → all keys (frames 0-{frames_to_show-1})", fontsize=10)
for i in range(1, frames_to_show):
    axes[1].axvline(x=i * Nw_vis - 0.5, color="black", linewidth=1)
axes[1].set_xlabel("← Frame 0 | Frame 1 | Frame 2 | Frame 3 →")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()
print("Block diagonal structure = within-frame attention is naturally stronger")
print("Decaying off-diagonal = temporally distant frames get less attention")

---
# 4) Token Compression for Video

Even with dynamic FPS, video produces many tokens. This section implements three
complementary compression strategies to further reduce the token count before the LLM.

| Strategy | Reduces | How |
|---|---|---|
| **Temporal pooling** | Frames | Average adjacent frame tokens |
| **Frame merging** | Frames | Weighted combination of similar frames |
| **Spatial token merging** (ToMe) | Patches/frame | Merge similar spatial tokens |

**Sample compression pipeline:**
```
Input:  40 frames × 64 patches = 2,560 tokens
  ↓ Temporal pooling (4× reduction):  10 groups × 64 patches = 640 tokens
  ↓ Spatial ToMe (2× reduction):      10 groups × 32 tokens  = 320 tokens

Final: 320 tokens (8× total reduction)
```

## 4.1 Temporal Pooling

The simplest temporal compression: group consecutive frames and average their token
representations. This assumes smooth temporal evolution within each group.

In [ ]:
class TemporalPooling(nn.Module):
    """Pool consecutive frames by averaging their token representations.

    Input:  (batch_num, num_frames, num_patches, model_dim)
    Output: (batch_num, num_frames // pool_size, num_patches, model_dim)

    If num_frames is not divisible by pool_size, the last group is smaller.
    """

    def __init__(self, pool_size: int = 4):
        super().__init__()
        self.pool_size = pool_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, N, D = x.shape
        ps = self.pool_size

        # Number of complete groups
        num_groups = (T + ps - 1) // ps

        # Pad if necessary to make T divisible by pool_size
        pad_t = num_groups * ps - T
        if pad_t > 0:
            x = F.pad(x, (0, 0, 0, 0, 0, pad_t))

        # (batch_num, num_groups, pool_size, num_patches, model_dim) → mean over pool_size
        x = x.reshape(B, num_groups, ps, N, D).mean(dim=2)
        return x


# Test temporal pooling
tp = TemporalPooling(pool_size=4)
x_in = torch.randn(2, 16, 64, 256)
x_out = tp(x_in)
print(f"Temporal pooling (pool_size=4):")
print(f"  Input:  {x_in.shape} → (batch_num, num_frames, num_patches, model_dim)")
print(f"  Output: {x_out.shape} → {16//4}× fewer frames")

## 4.2 Spatial Token Merging (ToMe-style)

Inspired by Token Merging (Bolya et al., 2023): within each frame, find the most similar
pairs of tokens and merge them by averaging. This is especially effective for video
where large spatial regions (background, sky) are nearly identical across patches.

**Algorithm per frame:**
1. Partition tokens into two sets (alternating indices)
2. Compute cosine similarity between set A and set B
3. For each token in set A, find its most similar partner in set B
4. Merge the top-r most similar pairs by averaging
5. Return the reduced token set

In [ ]:
class SpatialTokenMerging(nn.Module):
    """Merge similar spatial tokens within each frame (ToMe-inspired).

    Input:  (batch_num, num_frames, num_patches, model_dim)
    Output: (batch_num, num_frames, num_patches - num_merge, model_dim)

    Design: We use bipartite matching to avoid the O(N³) cost of finding global
    optimal merges. The alternating partition is a simple, effective heuristic.
    """

    def __init__(self, num_merge: int = 16):
        super().__init__()
        self.num_merge = num_merge

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, N, D = x.shape
        r = min(self.num_merge, N // 2)

        # Process each frame independently (but batched over B*T)
        # (batch_num * num_frames, num_patches, model_dim)
        x_flat = x.reshape(B * T, N, D)

        # Bipartite partition: even-indexed vs odd-indexed tokens
        src = x_flat[:, 0::2]  # (B*T, N//2, D) — "source" set
        dst = x_flat[:, 1::2]  # (B*T, N_dst, D) — "destination" set
        N_src = src.shape[1]
        N_dst = dst.shape[1]

        # Cosine similarity between source and destination tokens
        # (B*T, N_src, D) @ (B*T, D, N_dst) → (B*T, N_src, N_dst)
        src_norm = F.normalize(src, dim=-1)
        dst_norm = F.normalize(dst, dim=-1)
        similarity = torch.bmm(src_norm, dst_norm.transpose(1, 2))

        # For each source token, find most similar destination
        # (B*T, N_src)
        max_sim, max_idx = similarity.max(dim=-1)

        # Select top-r most similar pairs to merge
        # (B*T, r)
        _, top_r_indices = max_sim.topk(r, dim=-1)

        # Build output: unmerged source tokens + merged pairs in destination
        results = []
        for b in range(B * T):
            merge_src_idx = top_r_indices[b]  # which source tokens to merge
            keep_src_mask = torch.ones(N_src, dtype=torch.bool)
            keep_src_mask[merge_src_idx] = False

            # Tokens that stay as-is
            kept_src = src[b][keep_src_mask]  # (N_src - r, D)

            # Merge selected source tokens into their destination partners
            dst_copy = dst[b].clone()
            for i in range(r):
                s_idx = merge_src_idx[i]
                d_idx = max_idx[b, s_idx]
                # Average the source and destination token
                dst_copy[d_idx] = (dst_copy[d_idx] + src[b, s_idx]) / 2.0

            # Combine: unmerged sources + updated destinations
            results.append(torch.cat([kept_src, dst_copy], dim=0))

        # (B*T, N - r, D) → (B, T, N - r, D)
        output = torch.stack(results).reshape(B, T, N - r, D)
        return output


# Test spatial token merging
stm = SpatialTokenMerging(num_merge=16)
x_in = torch.randn(2, 8, 64, 256)
x_out = stm(x_in)
print(f"Spatial token merging (merge 16 per frame):")
print(f"  Input:  {x_in.shape} → 64 patches per frame")
print(f"  Output: {x_out.shape} → {64-16} patches per frame")
print(f"  Token reduction: {(1 - x_out.shape[2]/x_in.shape[2])*100:.0f}%")

## 4.3 Frame Merging

A temporal analogue of token merging: merge **entire frames** that are highly similar.
This is effective after the encoder, where similar frames have similar representations.
Weighted merging (vs. simple dropping) preserves more information.

In [ ]:
class FrameMerging(nn.Module):
    """Merge similar consecutive frames by weighted averaging.

    Input:  (batch_num, num_frames, num_patches, model_dim)
    Output: (batch_num, num_frames - num_merge, num_patches, model_dim)

    Strategy: Compute cosine similarity between consecutive frame representations
    (averaged over patches), then merge the most similar consecutive pairs.
    """

    def __init__(self, num_merge: int = 4):
        super().__init__()
        self.num_merge = num_merge

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, N, D = x.shape
        r = min(self.num_merge, T - 1)

        # Compute per-frame representation by averaging over patches
        # (batch_num, num_frames, model_dim)
        frame_repr = x.mean(dim=2)

        # Cosine similarity between consecutive frames
        # (batch_num, num_frames - 1)
        sim = F.cosine_similarity(frame_repr[:, :-1], frame_repr[:, 1:], dim=-1)

        results = []
        for b in range(B):
            # Find r most similar consecutive pairs
            _, merge_positions = sim[b].topk(r)
            merge_set = set(merge_positions.tolist())

            merged_frames = []
            i = 0
            while i < T:
                if i in merge_set and i + 1 < T:
                    # Merge frame i and i+1 with equal weight
                    merged = (x[b, i] + x[b, i + 1]) / 2.0
                    merged_frames.append(merged)
                    i += 2
                else:
                    merged_frames.append(x[b, i])
                    i += 1

            results.append(torch.stack(merged_frames))

        # Pad to same length within batch (different items may merge differently)
        max_len = max(r.shape[0] for r in results)
        padded = torch.zeros(B, max_len, N, D, device=x.device)
        for b, r_tensor in enumerate(results):
            padded[b, :r_tensor.shape[0]] = r_tensor

        return padded


# Test frame merging
fm = FrameMerging(num_merge=4)
x_in = torch.randn(1, 16, 64, 256)
x_out = fm(x_in)
print(f"Frame merging (merge 4 pairs):")
print(f"  Input:  {x_in.shape} → 16 frames")
print(f"  Output: {x_out.shape}")

## 4.4 Combined Compression Pipeline

In [ ]:
class VideoTokenCompressor(nn.Module):
    """Pipeline: Temporal Pooling → Spatial Token Merging.

    Input:  (batch_num, num_frames, num_patches, model_dim)
    Output: (batch_num, reduced_frames, reduced_patches, model_dim)

    Design: We apply temporal compression first (reducing frames) then spatial
    compression (reducing patches per frame). This order is more efficient because
    spatial merging's per-frame cost scales with the number of frames.
    """

    def __init__(self, temporal_pool: int = 4, spatial_merge: int = 16):
        super().__init__()
        self.temporal = TemporalPooling(pool_size=temporal_pool)
        self.spatial = SpatialTokenMerging(num_merge=spatial_merge)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T_in, N_in = x.shape[1], x.shape[2]
        x = self.temporal(x)
        x = self.spatial(x)
        T_out, N_out = x.shape[1], x.shape[2]
        return x


compressor = VideoTokenCompressor(temporal_pool=4, spatial_merge=16)
x_in = torch.randn(1, 32, 64, 256)
x_out = compressor(x_in)
total_in = x_in.shape[1] * x_in.shape[2]
total_out = x_out.shape[1] * x_out.shape[2]
print(f"Full compression pipeline:")
print(f"  Input:  {x_in.shape} → {total_in} total tokens")
print(f"  Output: {x_out.shape} → {total_out} total tokens")
print(f"  Compression: {total_in/total_out:.1f}× reduction")

---
# 5) Text-Timestamp Alignment (Qwen3-VL Style)

## 5.1 Motivation

Standard video-language models answer questions about video content but can't point to
**when** something happened. Qwen3-VL introduces text-timestamp alignment:

```
Input:   Video + "When does the player score?"
Output:  "The player scores a goal at 00:03:42"

Input:   Video + "Describe what happens between 01:00 and 02:00"
Output:  "A person walks into the room and sits down at the desk"
```

**How it works:**
1. Each visual token carries a timestamp (from mRoPE's time dimension)
2. Special `<time>` tokens in the vocabulary encode absolute timestamps
3. The model learns to align generated `<time>` tokens with the correct visual tokens
4. A timestamp prediction head converts visual features → timestamp logits

**Key insight:** The temporal grounding is bidirectional:
- **Text → Time:** "When does X happen?" → model generates timestamp tokens
- **Time → Text:** "Describe t=30s to t=45s" → model conditions on those frames

In [ ]:
class TimestampTokenizer:
    """Encode and decode timestamps as special tokens.

    Timestamps are quantized to a fixed resolution (e.g., 0.5s intervals).
    Each quantized timestamp gets a unique token ID in a reserved range.

    Example (resolution=0.5s, max_duration=60s):
        00:00.0 → token 0
        00:00.5 → token 1
        00:01.0 → token 2
        ...
        01:00.0 → token 120
    """

    def __init__(self, max_duration_sec: float = 60.0, resolution_sec: float = 0.5):
        self.max_duration = max_duration_sec
        self.resolution = resolution_sec
        self.num_bins = int(max_duration_sec / resolution_sec) + 1

    def encode(self, timestamp_sec: float) -> int:
        """Convert a timestamp (seconds) to a bin index."""
        bin_idx = int(round(timestamp_sec / self.resolution))
        return min(bin_idx, self.num_bins - 1)

    def decode(self, bin_idx: int) -> float:
        """Convert a bin index back to timestamp (seconds)."""
        return bin_idx * self.resolution

    def format_timestamp(self, timestamp_sec: float) -> str:
        """Human-readable timestamp string."""
        minutes = int(timestamp_sec) // 60
        seconds = timestamp_sec % 60
        return f"{minutes:02d}:{seconds:05.2f}"


ts_tokenizer = TimestampTokenizer(max_duration_sec=60.0, resolution_sec=0.5)
print(f"Timestamp vocabulary size: {ts_tokenizer.num_bins} bins")
print(f"Resolution: {ts_tokenizer.resolution}s")

# Demo encoding/decoding
for t in [0.0, 3.7, 15.0, 42.3, 59.9]:
    bin_id = ts_tokenizer.encode(t)
    decoded = ts_tokenizer.decode(bin_id)
    print(f"  {ts_tokenizer.format_timestamp(t)} → bin {bin_id} → decoded {ts_tokenizer.format_timestamp(decoded)}")

## 5.2 Timestamp Prediction Head

Given visual token features, predict which timestamp bin each token corresponds to.
This is used during generation to produce `<time>` tokens that are grounded in the video.

In [ ]:
class TimestampPredictor(nn.Module):
    """Predict timestamp bins from visual or cross-modal features.

    Input:  (batch_num, seq_len, model_dim) — features from the cross-attention layer
    Output: (batch_num, seq_len, num_time_bins) — logits over timestamp bins

    Architecture: Two-layer MLP with GELU, projecting to num_time_bins.
    We use a separate head (not shared with the LM head) because timestamp
    prediction is fundamentally a regression-like task quantized into bins.
    """

    def __init__(self, model_dim: int, num_time_bins: int, hidden_dim: int = 512):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(model_dim),
            nn.Linear(model_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_time_bins),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            features: (batch_num, seq_len, model_dim)
        Returns:
            logits: (batch_num, seq_len, num_time_bins)
        """
        return self.head(features)


# Test
ts_pred = TimestampPredictor(model_dim=256, num_time_bins=ts_tokenizer.num_bins)
feat = torch.randn(2, 10, 256)
logits = ts_pred(feat)
print(f"Timestamp predictor:")
print(f"  Input:  {feat.shape} → (batch_num, seq_len, model_dim)")
print(f"  Output: {logits.shape} → (batch_num, seq_len, num_time_bins)")

# Simulate prediction
pred_bins = logits[0].argmax(dim=-1)
print(f"\nPredicted timestamps for batch 0:")
for i, b in enumerate(pred_bins[:5]):
    t = ts_tokenizer.decode(b.item())
    print(f"  Token {i}: bin {b.item()} → {ts_tokenizer.format_timestamp(t)}")

## 5.3 Temporal Grounding Loss

To train the timestamp predictor, we need a loss that encourages correct temporal alignment.
We use a **soft cross-entropy** with Gaussian smoothing around the ground-truth bin, because
adjacent timestamps are nearly equivalent (predicting 3.0s when truth is 3.5s is almost right).

```
Ground truth: t = 5.0s → bin 10
Hard target:  [0, 0, ..., 0, 1, 0, ..., 0]  (one-hot at bin 10)
Soft target:  [0, 0, ..., 0.05, 0.24, 0.42, 0.24, 0.05, ..., 0]  (Gaussian around bin 10)
```

In [ ]:
def temporal_grounding_loss(
    pred_logits: torch.Tensor,
    target_bins: torch.Tensor,
    num_bins: int,
    sigma: float = 2.0,
) -> torch.Tensor:
    """Compute soft cross-entropy loss for timestamp prediction.

    Args:
        pred_logits: (batch_num, seq_len, num_bins)
        target_bins: (batch_num, seq_len) — ground-truth bin indices
        num_bins: total number of timestamp bins
        sigma: std dev of the Gaussian smoothing (in bins)

    Returns:
        loss: scalar — average cross-entropy with soft labels
    """
    B, S, _ = pred_logits.shape

    # Build soft target distribution: Gaussian centered on the true bin
    # (num_bins,) — bin indices
    bin_indices = torch.arange(num_bins, device=pred_logits.device, dtype=torch.float32)
    # (batch_num, seq_len, 1) - (num_bins,) → (batch_num, seq_len, num_bins)
    targets_expanded = target_bins.unsqueeze(-1).float()
    soft_targets = torch.exp(-0.5 * ((bin_indices - targets_expanded) / sigma) ** 2)
    # Normalize to probability distribution
    soft_targets = soft_targets / soft_targets.sum(dim=-1, keepdim=True)

    # Cross-entropy with soft labels
    log_probs = F.log_softmax(pred_logits, dim=-1)
    # (batch_num, seq_len, num_bins) → scalar
    loss = -(soft_targets * log_probs).sum(dim=-1).mean()

    return loss


# Demo
pred = torch.randn(2, 5, ts_tokenizer.num_bins)
target = torch.tensor([[10, 20, 30, 40, 50], [5, 15, 25, 35, 45]])
loss = temporal_grounding_loss(pred, target, ts_tokenizer.num_bins, sigma=2.0)
print(f"Temporal grounding loss: {loss.item():.4f}")

# Visualize soft target
target_bin = 20
bins = torch.arange(ts_tokenizer.num_bins)
soft = torch.exp(-0.5 * ((bins.float() - target_bin) / 2.0) ** 2)
soft = soft / soft.sum()
plt.figure(figsize=(10, 3))
plt.bar(bins[:40].numpy(), soft[:40].numpy(), alpha=0.7)
plt.axvline(x=target_bin, color="red", linestyle="--", label=f"True bin={target_bin}")
plt.xlabel("Timestamp bin")
plt.ylabel("Target probability")
plt.title("Soft Temporal Grounding Target (σ=2.0)")
plt.legend()
plt.tight_layout()
plt.show()

---
# 6) Building a Mini Video-Language Model

## 6.1 Architecture Overview

We combine all components into a complete video-language model:

```
Raw Video (T, C, H, W)
    │
    ├─ Dynamic FPS Sampling → (T', C, H, W)   [Section 1.4]
    │
    ├─ Patch Embedding → (T', N, D)            [Section 2.2]
    │
    ├─ mRoPE positions → (time, height, width) [Section 3]
    │
    ├─ Factorized Encoder (L layers) → (T', N, D)  [Section 2.4]
    │
    ├─ Token Compression → (T'', N', D)        [Section 4]
    │
    ├─ Visual Projector → (T''×N', D_llm)      [Linear projection to LLM dim]
    │
    ├─ [VIS_TOKENS] + [TEXT_TOKENS]             [Concatenate with text]
    │
    └─ LLM Decoder → text + timestamps         [Sections 5 + 6]
```

For this educational notebook, our "LLM" is a small 2-layer transformer decoder.
In production, this would be Qwen2.5 or similar.

In [ ]:
@dataclass
class VideoLMConfig:
    """Configuration for the mini video-language model."""
    # Vision encoder
    img_size: int = 64
    patch_size: int = 8
    in_channels: int = 3
    vision_dim: int = 256
    vision_heads: int = 8
    vision_layers: int = 2
    attention_type: str = "factorized"  # "factorized" or "joint"

    # Compression
    temporal_pool: int = 2
    spatial_merge: int = 8

    # Language model
    vocab_size: int = 1000
    llm_dim: int = 256
    llm_heads: int = 8
    llm_layers: int = 2
    max_seq_len: int = 512

    # Timestamp
    max_duration_sec: float = 60.0
    timestamp_resolution: float = 0.5

    # Video sampling
    min_fps: float = 0.5
    max_fps: float = 8.0
    original_fps: float = 30.0


config = VideoLMConfig()
print("VideoLM Configuration:")
for field in config.__dataclass_fields__:
    print(f"  {field}: {getattr(config, field)}")

## 6.2 Vision Encoder

Stacks multiple factorized (or joint) attention blocks to build rich video representations.

In [ ]:
class VideoEncoder(nn.Module):
    """Video encoder: patch embedding + stacked spatial-temporal attention.

    Input:  (batch_num, num_frames, C, H, W)
    Output: (batch_num, num_frames, num_patches, vision_dim)

    The encoder outputs one token per patch per frame, enriched with both
    spatial context (what's around this patch) and temporal context (what
    happened at this position in other frames).
    """

    def __init__(self, config: VideoLMConfig):
        super().__init__()
        self.patch_embed = VideoPatchEmbedding(
            config.img_size, config.patch_size, config.in_channels, config.vision_dim
        )
        num_patches = self.patch_embed.num_patches

        # Learnable CLS token (one per frame) for frame-level representations
        self.cls_token = nn.Parameter(torch.randn(1, 1, 1, config.vision_dim) * 0.02)

        # mRoPE for 3D positional encoding
        self.mrope = VideoMRoPE(head_dim=config.vision_dim // config.vision_heads)

        # Stacked attention blocks
        if config.attention_type == "factorized":
            self.blocks = nn.ModuleList([
                FactorizedTransformerBlock(config.vision_dim, config.vision_heads)
                for _ in range(config.vision_layers)
            ])
        else:
            self.blocks = nn.ModuleList([
                JointSpaceTimeBlock(config.vision_dim, config.vision_heads)
                for _ in range(config.vision_layers)
            ])

        self.attention_type = config.attention_type
        self.norm = nn.LayerNorm(config.vision_dim)
        self.num_patches_h = self.patch_embed.num_patches_h
        self.num_patches_w = self.patch_embed.num_patches_w

    def _build_position_ids(self, B: int, T: int, N: int, device: torch.device):
        """Build (time, height, width) position IDs for mRoPE."""
        nh, nw = self.num_patches_h, self.num_patches_w
        time_ids = torch.arange(T, device=device).repeat_interleave(N).unsqueeze(0).expand(B, -1)
        h_ids = (torch.arange(N, device=device) // nw).repeat(T).unsqueeze(0).expand(B, -1)
        w_ids = (torch.arange(N, device=device) % nw).repeat(T).unsqueeze(0).expand(B, -1)
        return time_ids, h_ids, w_ids

    def forward(self, video: torch.Tensor) -> Tuple[torch.Tensor, Dict]:
        """
        Args:
            video: (batch_num, num_frames, C, H, W)
        Returns:
            features: (batch_num, num_frames, num_patches, vision_dim)
            info: dict with attention weights and position IDs
        """
        B, T = video.shape[:2]

        # Patch embedding
        # (batch_num, num_frames, C, H, W) → (batch_num, num_frames, num_patches, vision_dim)
        x = self.patch_embed(video)
        N = x.shape[2]

        # Build position IDs for mRoPE (stored but used if needed by downstream)
        time_ids, h_ids, w_ids = self._build_position_ids(B, T, N, video.device)

        # Apply transformer blocks
        attn_info = {}
        for i, block in enumerate(self.blocks):
            x, attn = block(x)
            attn_info[f"layer_{i}"] = attn

        x = self.norm(x)
        info = {"attn": attn_info, "time_ids": time_ids, "h_ids": h_ids, "w_ids": w_ids}
        return x, info


# Test
encoder = VideoEncoder(config)
test_vid = torch.randn(2, 8, 3, 64, 64)
enc_out, enc_info = encoder(test_vid)
print(f"Video encoder:")
print(f"  Input:  {test_vid.shape} → (batch_num, num_frames, C, H, W)")
print(f"  Output: {enc_out.shape} → (batch_num, num_frames, num_patches, vision_dim)")
total_params = sum(p.numel() for p in encoder.parameters())
print(f"  Parameters: {total_params:,}")

## 6.3 Visual Projector

Bridges the vision encoder's output space to the LLM's input space. In practice,
this is often a 2-layer MLP (LLaVA-style) or a more complex perceiver resampler.
We use a simple MLP for clarity.

In [ ]:
class VisualProjector(nn.Module):
    """Project vision tokens to the LLM's embedding space.

    Input:  (batch_num, num_tokens, vision_dim)
    Output: (batch_num, num_tokens, llm_dim)

    Design: 2-layer MLP with GELU (following LLaVA). More sophisticated
    approaches like Q-Former (BLIP-2) or perceiver resampler (Flamingo)
    also change the number of tokens, but MLP projection is simpler and
    competitive when combined with good token compression.
    """

    def __init__(self, vision_dim: int, llm_dim: int):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
            nn.LayerNorm(llm_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)

## 6.4 Causal Transformer Decoder (Mini LLM)

A small causal transformer that processes the concatenated visual + text tokens
and generates text autoregressively. This stands in for a full LLM.

In [ ]:
class CausalDecoderBlock(nn.Module):
    """Single block of a causal transformer decoder.

    Input:  (batch_num, seq_len, model_dim)
    Output: (batch_num, seq_len, model_dim)

    Uses causal masking so position i can only attend to positions <= i.
    """

    def __init__(self, model_dim: int, num_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(model_dim)
        self.attn = MultiHeadSelfAttention(model_dim, num_heads, dropout)
        mlp_dim = int(model_dim * mlp_ratio)
        self.norm2 = nn.LayerNorm(model_dim)
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, model_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        S = x.shape[1]
        # Build causal mask: upper triangle = -inf
        causal_mask = torch.triu(torch.ones(S, S, device=x.device) * float("-inf"), diagonal=1)
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, attn_mask=causal_mask)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x


class MiniLLM(nn.Module):
    """Minimal causal language model for the video-language pipeline.

    Input:  (batch_num, seq_len, llm_dim)  — pre-embedded tokens
    Output: (batch_num, seq_len, vocab_size) — logits

    This is intentionally tiny (2 layers). In production, replace with
    a pretrained LLM and freeze or LoRA-tune.
    """

    def __init__(self, config: VideoLMConfig):
        super().__init__()
        self.text_embed = nn.Embedding(config.vocab_size, config.llm_dim)
        self.blocks = nn.ModuleList([
            CausalDecoderBlock(config.llm_dim, config.llm_heads)
            for _ in range(config.llm_layers)
        ])
        self.norm = nn.LayerNorm(config.llm_dim)
        self.lm_head = nn.Linear(config.llm_dim, config.vocab_size, bias=False)

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        """
        Args:
            embeddings: (batch_num, seq_len, llm_dim) — already embedded (visual + text)
        Returns:
            logits: (batch_num, seq_len, vocab_size)
        """
        x = embeddings
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.lm_head(x)

## 6.5 Full Video-Language Model

Integrates everything: dynamic FPS → encoder → compressor → projector → LLM → timestamps.

In [ ]:
class VideoLanguageModel(nn.Module):
    """Complete video-language model with temporal grounding.

    Pipeline:
      1. Sample frames (dynamic FPS or uniform)
      2. Encode with spatial-temporal attention
      3. Compress tokens (temporal pooling + spatial merging)
      4. Project to LLM space
      5. Concatenate with text token embeddings
      6. Decode with causal LLM
      7. Optionally predict timestamps

    Input:  video (B, T_raw, C, H, W) + text_ids (B, S_text)
    Output: text logits (B, S_total, vocab_size) + optional timestamp logits
    """

    def __init__(self, config: VideoLMConfig):
        super().__init__()
        self.config = config

        # Vision pipeline
        self.encoder = VideoEncoder(config)
        self.compressor = VideoTokenCompressor(config.temporal_pool, config.spatial_merge)
        self.projector = VisualProjector(config.vision_dim, config.llm_dim)

        # Language model
        self.llm = MiniLLM(config)

        # Timestamp prediction
        ts_tokenizer = TimestampTokenizer(config.max_duration_sec, config.timestamp_resolution)
        self.ts_predictor = TimestampPredictor(config.llm_dim, ts_tokenizer.num_bins)
        self.ts_tokenizer = ts_tokenizer

        # Special token embeddings
        self.vis_start = nn.Parameter(torch.randn(1, 1, config.llm_dim) * 0.02)
        self.vis_end = nn.Parameter(torch.randn(1, 1, config.llm_dim) * 0.02)

    def encode_video(self, video: torch.Tensor) -> Tuple[torch.Tensor, Dict]:
        """Encode video frames into compressed visual tokens.

        Args:
            video: (batch_num, num_frames, C, H, W)
        Returns:
            visual_tokens: (batch_num, num_vis_tokens, llm_dim)
            info: dict with encoder details
        """
        # Encode: (B, T, C, H, W) → (B, T, N, vision_dim)
        features, info = self.encoder(video)

        # Compress: (B, T, N, D) → (B, T', N', D)
        features = self.compressor(features)

        B, T, N, D = features.shape
        # Flatten space-time: (B, T'*N', D)
        features = features.reshape(B, T * N, D)

        # Project to LLM space: (B, T'*N', vision_dim) → (B, T'*N', llm_dim)
        visual_tokens = self.projector(features)

        info["num_vis_tokens"] = visual_tokens.shape[1]
        return visual_tokens, info

    def forward(
        self,
        video: torch.Tensor,
        text_ids: torch.Tensor,
        return_timestamps: bool = False,
    ) -> Dict[str, torch.Tensor]:
        """
        Args:
            video: (batch_num, num_frames, C, H, W)
            text_ids: (batch_num, text_seq_len) — token IDs
            return_timestamps: whether to compute timestamp logits

        Returns:
            dict with:
                'logits': (batch_num, total_seq_len, vocab_size)
                'ts_logits': (batch_num, total_seq_len, num_time_bins)  [if return_timestamps]
                'num_vis_tokens': int
        """
        B = video.shape[0]

        # Encode and compress video
        visual_tokens, info = self.encode_video(video)

        # Embed text tokens
        # (batch_num, text_seq_len) → (batch_num, text_seq_len, llm_dim)
        text_embeds = self.llm.text_embed(text_ids)

        # Concatenate: [VIS_START] + visual_tokens + [VIS_END] + text_tokens
        vis_start = self.vis_start.expand(B, -1, -1)
        vis_end = self.vis_end.expand(B, -1, -1)
        combined = torch.cat([vis_start, visual_tokens, vis_end, text_embeds], dim=1)

        # LLM forward pass
        # (batch_num, total_seq_len, llm_dim) → (batch_num, total_seq_len, vocab_size)
        logits = self.llm(combined)

        output = {
            "logits": logits,
            "num_vis_tokens": info["num_vis_tokens"],
        }

        if return_timestamps:
            # Get hidden states from the LLM's penultimate layer for timestamp prediction
            # For simplicity, reuse the logits' input (before lm_head)
            x = combined
            for block in self.llm.blocks:
                x = block(x)
            x = self.llm.norm(x)
            output["ts_logits"] = self.ts_predictor(x)

        return output


# Test the full model
model = VideoLanguageModel(config)
test_video = torch.randn(2, 16, 3, 64, 64)
test_text = torch.randint(0, config.vocab_size, (2, 20))

output = model(test_video, test_text, return_timestamps=True)
print(f"Video-Language Model (full pipeline):")
print(f"  Video input:  {test_video.shape}")
print(f"  Text input:   {test_text.shape}")
print(f"  Visual tokens: {output['num_vis_tokens']}")
print(f"  LM logits:    {output['logits'].shape} → (batch_num, total_seq, vocab_size)")
print(f"  TS logits:    {output['ts_logits'].shape} → (batch_num, total_seq, num_time_bins)")
total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_params:,}")

---
# 7) Demo: Video QA with Temporal Grounding

## 7.1 Training on Synthetic Data

We create a tiny synthetic dataset where:
- Videos have events at specific timestamps
- Questions ask "When does X happen?" or "What happens at time T?"
- The model must learn to associate visual events with timestamps

This demonstrates the temporal grounding pipeline end-to-end.

In [ ]:
def create_synthetic_qa_dataset(
    num_samples: int = 32,
    num_frames: int = 16,
    height: int = 64,
    width: int = 64,
    vocab_size: int = 1000,
    max_text_len: int = 20,
) -> Dict[str, torch.Tensor]:
    """Create a synthetic video QA dataset for training/testing.

    Each video has a 'flash event' at a random frame. The question tokens
    encode a query, and the target is the correct timestamp bin.

    Returns:
        dict with 'videos', 'text_ids', 'target_ids', 'event_frames', 'event_timestamps'
    """
    videos = torch.zeros(num_samples, num_frames, 3, height, width)
    text_ids = torch.randint(10, vocab_size, (num_samples, max_text_len))
    target_ids = torch.randint(10, vocab_size, (num_samples, max_text_len))
    event_frames = torch.zeros(num_samples, dtype=torch.long)
    event_timestamps = torch.zeros(num_samples)

    ts_tok = TimestampTokenizer(max_duration_sec=60.0, resolution_sec=0.5)
    fps = 30.0

    for i in range(num_samples):
        # Random event frame
        ef = torch.randint(1, num_frames - 1, (1,)).item()
        event_frames[i] = ef
        event_timestamps[i] = ef / fps

        # Background: dark frames with slight noise
        videos[i] = torch.rand(num_frames, 3, height, width) * 0.1

        # Event: bright flash on the event frame
        videos[i, ef] = torch.rand(3, height, width) * 0.5 + 0.5

        # Text: use a simple pattern (token 5 = "when", token 6 = "flash")
        text_ids[i, 0] = 5  # "when"
        text_ids[i, 1] = 6  # "does"
        text_ids[i, 2] = 7  # "flash"
        text_ids[i, 3] = 8  # "happen"
        text_ids[i, 4] = 9  # "?"

    return {
        "videos": videos,
        "text_ids": text_ids,
        "target_ids": target_ids,
        "event_frames": event_frames,
        "event_timestamps": event_timestamps,
    }


dataset = create_synthetic_qa_dataset(num_samples=64, num_frames=16)
print("Synthetic QA dataset:")
for k, v in dataset.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape} (dtype={v.dtype})")

## 7.2 Training Loop

A minimal training loop demonstrating both language modeling loss and
temporal grounding loss working together.

In [ ]:
def train_video_lm(
    model: VideoLanguageModel,
    dataset: Dict[str, torch.Tensor],
    num_epochs: int = 5,
    batch_size: int = 8,
    lr: float = 1e-3,
) -> List[float]:
    """Train the video-language model on synthetic QA data.

    Loss = LM_loss (next-token prediction) + λ * timestamp_loss (temporal grounding)
    """
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    videos = dataset["videos"]
    text_ids = dataset["text_ids"]
    target_ids = dataset["target_ids"]
    event_timestamps = dataset["event_timestamps"]
    num_samples = videos.shape[0]

    ts_tokenizer = model.ts_tokenizer
    losses = []

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0

        # Shuffle
        perm = torch.randperm(num_samples)
        for start in range(0, num_samples - batch_size + 1, batch_size):
            idx = perm[start : start + batch_size]
            batch_vid = videos[idx]
            batch_text = text_ids[idx]
            batch_target = target_ids[idx]
            batch_ts = event_timestamps[idx]

            output = model(batch_vid, batch_text, return_timestamps=True)

            # Language modeling loss on text portion
            # Shift logits and targets for next-token prediction
            num_vis = output["num_vis_tokens"] + 2  # +2 for start/end tokens
            text_logits = output["logits"][:, num_vis:-1]
            text_targets = batch_target[:, 1:]
            min_len = min(text_logits.shape[1], text_targets.shape[1])
            lm_loss = F.cross_entropy(
                text_logits[:, :min_len].reshape(-1, config.vocab_size),
                text_targets[:, :min_len].reshape(-1),
            )

            # Timestamp grounding loss on visual tokens
            ts_logits = output["ts_logits"][:, 1 : num_vis - 1]  # visual tokens only
            target_bins = torch.stack([
                torch.full((ts_logits.shape[1],), ts_tokenizer.encode(t.item()))
                for t in batch_ts
            ])
            ts_loss = temporal_grounding_loss(
                ts_logits, target_bins, ts_tokenizer.num_bins, sigma=2.0
            )

            # Combined loss
            loss = lm_loss + 0.5 * ts_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / max(num_batches, 1)
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs} — Loss: {avg_loss:.4f}")

    return losses


model = VideoLanguageModel(config)
losses = train_video_lm(model, dataset, num_epochs=8, batch_size=8, lr=1e-3)

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(range(1, len(losses) + 1), losses, marker="o", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss — Video QA with Temporal Grounding")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7.3 Inference: "What happens at the 2-minute mark?"

We demonstrate the model's output structure. With our tiny model and synthetic data,
the outputs won't be semantically meaningful, but the pipeline correctly routes
video features through encoding → compression → projection → decoding → timestamp prediction.

In [ ]:
@torch.no_grad()
def video_qa_inference(
    model: VideoLanguageModel,
    video: torch.Tensor,
    question_ids: torch.Tensor,
    max_new_tokens: int = 10,
) -> Dict:
    """Run inference on a single video + question.

    Args:
        model: trained VideoLanguageModel
        video: (1, T, C, H, W) — single video
        question_ids: (1, S) — tokenized question
        max_new_tokens: how many tokens to generate

    Returns:
        dict with generated token IDs and predicted timestamps
    """
    model.eval()

    output = model(video, question_ids, return_timestamps=True)
    logits = output["logits"]
    ts_logits = output["ts_logits"]

    # Get the last token's prediction as the "answer start"
    generated_ids = []
    for step in range(max_new_tokens):
        next_token = logits[0, -1].argmax().item()
        generated_ids.append(next_token)

    # Get timestamp predictions for visual tokens
    num_vis = output["num_vis_tokens"]
    vis_ts_logits = ts_logits[0, 1 : num_vis + 1]  # skip VIS_START
    pred_bins = vis_ts_logits.argmax(dim=-1)
    pred_times = [model.ts_tokenizer.decode(b.item()) for b in pred_bins]

    return {
        "generated_ids": generated_ids,
        "pred_timestamps": pred_times,
        "num_vis_tokens": num_vis,
    }


# Run inference on a test sample
test_vid = dataset["videos"][:1]
test_q = dataset["text_ids"][:1]
true_event = dataset["event_timestamps"][0].item()

result = video_qa_inference(model, test_vid, test_q)
ts_tok = TimestampTokenizer()

print(f"Video QA Inference Demo")
print(f"{'='*50}")
print(f"True event time: {ts_tok.format_timestamp(true_event)}")
print(f"Visual tokens: {result['num_vis_tokens']}")
print(f"Generated token IDs: {result['generated_ids']}")
print(f"\nPredicted timestamps for visual tokens (first 10):")
for i, t in enumerate(result["pred_timestamps"][:10]):
    print(f"  Visual token {i}: {ts_tok.format_timestamp(t)}")

---
# 8) Bonus: Scaling to Long Videos

## 8.1 Hour-Long Video Understanding

Processing hour-long videos is an open research challenge. The approaches we've built
provide a foundation, but additional techniques are needed at this scale:

```
1-hour video @ 30fps = 108,000 frames
Even at 1 FPS:  3,600 frames × 256 patches = 921,600 tokens → still too many

Solution stack:
  ├─ Dynamic FPS:           3,600 → ~500 frames    (7× reduction)
  ├─ Aggressive temporal pool: 500 → ~50 groups     (10× reduction)
  ├─ Spatial token merging:    256 → 64 patches     (4× reduction)
  └─ Final: 50 × 64 = 3,200 tokens                 (288× total reduction)
```

## 8.2 Hierarchical Encoding

For very long videos, a two-stage approach works well:
1. **Segment-level:** Encode 10-second chunks independently
2. **Video-level:** A lightweight temporal transformer over segment summaries

In [ ]:
class HierarchicalVideoEncoder(nn.Module):
    """Two-stage encoder for long video understanding.

    Stage 1: Encode short segments (e.g., 10s) with the full video encoder
    Stage 2: Aggregate segment representations with a lightweight temporal transformer

    Input:  (batch_num, total_frames, C, H, W) — potentially thousands of frames
    Output: (batch_num, num_segments, model_dim) — one token per segment

    This reduces a 1-hour video to ~360 tokens (at 10s segments), which is
    easily processable by an LLM.
    """

    def __init__(self, config: VideoLMConfig, segment_frames: int = 16):
        super().__init__()
        self.segment_frames = segment_frames
        self.encoder = VideoEncoder(config)
        self.compressor = VideoTokenCompressor(config.temporal_pool, config.spatial_merge)

        # Segment-level aggregation: average pool then project
        self.segment_proj = nn.Sequential(
            nn.LayerNorm(config.vision_dim),
            nn.Linear(config.vision_dim, config.vision_dim),
            nn.GELU(),
        )

        # Lightweight cross-segment temporal transformer
        self.cross_segment = nn.ModuleList([
            CausalDecoderBlock(config.vision_dim, config.vision_heads)
            for _ in range(2)
        ])
        self.final_norm = nn.LayerNorm(config.vision_dim)

    def forward(self, video: torch.Tensor) -> torch.Tensor:
        """
        Args:
            video: (batch_num, total_frames, C, H, W)
        Returns:
            segment_features: (batch_num, num_segments, vision_dim)
        """
        B, T_total, C, H, W = video.shape
        sf = self.segment_frames

        # Split into segments
        num_segments = (T_total + sf - 1) // sf
        pad_t = num_segments * sf - T_total
        if pad_t > 0:
            video = F.pad(video, (0, 0, 0, 0, 0, 0, 0, pad_t))

        # (batch_num, num_segments, segment_frames, C, H, W)
        segments = video.reshape(B, num_segments, sf, C, H, W)

        # Encode each segment
        segment_features = []
        for s in range(num_segments):
            # (batch_num, segment_frames, C, H, W) → (batch_num, T', N', vision_dim)
            enc_out, _ = self.encoder(segments[:, s])
            compressed = self.compressor(enc_out)
            # Average pool to single vector per segment
            # (batch_num, T'*N', vision_dim) → (batch_num, vision_dim)
            seg_feat = compressed.reshape(B, -1, compressed.shape[-1]).mean(dim=1)
            segment_features.append(seg_feat)

        # (batch_num, num_segments, vision_dim)
        x = torch.stack(segment_features, dim=1)
        x = self.segment_proj(x)

        # Cross-segment temporal reasoning
        for block in self.cross_segment:
            x = block(x)
        x = self.final_norm(x)

        return x


# Test hierarchical encoder
hier_encoder = HierarchicalVideoEncoder(config, segment_frames=8)
long_video = torch.randn(1, 48, 3, 64, 64)  # Simulating a longer video with 48 frames
hier_out = hier_encoder(long_video)
print(f"Hierarchical encoder:")
print(f"  Input: {long_video.shape} → 48 frames")
print(f"  Output: {hier_out.shape} → {hier_out.shape[1]} segment tokens")
print(f"  Compression: {48 * (64//8)**2} raw tokens → {hier_out.shape[1]} tokens")

## 8.3 Streaming Video Processing

For real-time applications (live streams, video calls), we can't wait for the entire
video. Instead, we process frames as they arrive in a sliding window.

In [ ]:
class StreamingVideoProcessor:
    """Process video frames in a streaming fashion with a sliding window.

    Maintains a buffer of recent frames and encoder states. When the buffer
    reaches window_size, encodes and compresses, then shifts the window.

    This simulates real-time video understanding where frames arrive one at a time.
    """

    def __init__(
        self,
        encoder: VideoEncoder,
        compressor: VideoTokenCompressor,
        window_size: int = 8,
        stride: int = 4,
    ):
        self.encoder = encoder
        self.compressor = compressor
        self.window_size = window_size
        self.stride = stride
        self.buffer: List[torch.Tensor] = []
        self.encoded_history: List[torch.Tensor] = []

    def reset(self):
        """Clear the frame buffer and encoded history."""
        self.buffer = []
        self.encoded_history = []

    @torch.no_grad()
    def process_frame(self, frame: torch.Tensor) -> Optional[torch.Tensor]:
        """Add a frame to the buffer. If window is full, encode and return features.

        Args:
            frame: (C, H, W) — single video frame

        Returns:
            features: (1, num_tokens, vision_dim) if window was processed, else None
        """
        self.buffer.append(frame)

        if len(self.buffer) >= self.window_size:
            # Stack buffer into a mini-video
            # (window_size, C, H, W) → (1, window_size, C, H, W)
            window = torch.stack(self.buffer[-self.window_size :]).unsqueeze(0)

            # Encode and compress
            features, _ = self.encoder(window)
            features = self.compressor(features)
            # (1, T', N', D) → (1, T'*N', D)
            features = features.reshape(1, -1, features.shape[-1])
            self.encoded_history.append(features)

            # Slide window by stride
            self.buffer = self.buffer[self.stride :]

            return features
        return None

    def get_full_context(self) -> Optional[torch.Tensor]:
        """Return all encoded history concatenated.

        Returns:
            (1, total_tokens, vision_dim) or None if nothing encoded yet
        """
        if not self.encoded_history:
            return None
        return torch.cat(self.encoded_history, dim=1)


# Demo streaming processing
stream = StreamingVideoProcessor(
    encoder=VideoEncoder(config),
    compressor=VideoTokenCompressor(temporal_pool=2, spatial_merge=8),
    window_size=8,
    stride=4,
)

print("Streaming video processing demo:")
print("=" * 50)
for i in range(30):
    frame = torch.randn(3, 64, 64)
    result = stream.process_frame(frame)
    if result is not None:
        ctx = stream.get_full_context()
        print(f"  Frame {i:3d}: Window processed → "
              f"{result.shape[1]} new tokens, "
              f"{ctx.shape[1]} total context tokens")
    else:
        print(f"  Frame {i:3d}: Buffering ({len(stream.buffer)}/{stream.window_size})")

---
# 9) Summary and Key Takeaways

## What we built (all from scratch):

| Component | Key Idea | Section |
|---|---|---|
| **Dynamic FPS** | Motion-adaptive sampling rate | 1.4 |
| **Factorized Attention** | Spatial-only + temporal-only (ViViT) | 2.4 |
| **Joint Space-Time Attention** | All patches × all frames | 2.5 |
| **3D mRoPE** | Rotary embeddings for (time, height, width) | 3 |
| **Token Compression** | Temporal pooling + spatial merging (ToMe) | 4 |
| **Text-Timestamp Alignment** | Temporal grounding with soft targets | 5 |
| **Video-Language Model** | Full pipeline: sample → encode → compress → decode | 6 |
| **Hierarchical Encoding** | Two-stage for hour-long videos | 8.2 |
| **Streaming Processing** | Sliding window for real-time | 8.3 |

## Key complexity results:

```
Joint attention:      O((T × N)²)     — best quality, expensive
Factorized attention: O(T·N² + N·T²)  — good quality, efficient
+ Dynamic FPS:        T_raw → T_eff   — 5-10× frame reduction
+ Token compression:  T×N → T'×N'     — additional 4-8× reduction
= Total:              76,800 tokens → ~300-600 tokens (100-250× reduction)
```

## Papers to read next:
1. **Qwen2.5-VL** — Dynamic FPS + mRoPE + native resolution processing
2. **Qwen3-VL** — Text-timestamp alignment for temporal grounding
3. **ViViT** (Arnab et al., 2021) — Factorized video transformer taxonomy
4. **Token Merging** (Bolya et al., 2023) — Spatial token compression
5. **LongVILA** — Hour-long video understanding with system-level optimizations
6. **VideoLLaMA 2** — Audio-visual language models for video